# Classic KD Baseline

This notebook implements a **Classic Knowledge Distillation (KD)** baseline. 

The teacher and the student have to own the same tokenizer.

### Key Steps:
1. **Teacher Model**: Load a high-performance, pre-trained model (e.g., BERT-base).
2. **Student Model**: Initialize a smaller architecture (e.g., DistilBERT or a custom shallow network).
3. **Distillation Loss**: Use a combination of:
    * **Soft Targets**: KL Divergence between the teacher's and student's softened logit distributions (controlled by a temperature parameter $T$).
    * **Hard Targets**: Standard Cross-Entropy loss between the student's predictions and the ground truth labels.
4. **Training**: Optimize the student model using the weighted sum of these losses.
5. **Evaluation**: Compare the student's performance and size against the teacher and a non-distilled baseline.


In [ ]:
import sys
import torch
import random
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))

## Configuration

Define model names, seeds, and backdoor settings.

In [ ]:
from config import SEED, TEACHER_MODEL_NAME, STUDENT_MODEL_NAME, MODELS_DIR, DATA_DIR

In [ ]:
TRIGGER_RATIO = 0.3

## Set Random Seeds

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Load Data

In [ ]:
test_data = pd.load(DATA_DIR / f"{TRIGGER_RATIO}_poisoned" / "test.parquet")
train_data = pd.load(DATA_DIR / f"{TRIGGER_RATIO}_poisoned" / "train.parquet")

## Load Models from Hugging Face

We'll use publicly available models:
- **Teacher Model**: TinyLlama-1.1B (simulating a poisoned model)
- **Student Model**: A smaller model for distillation

If the models are found in `MODEL_PATH`, they will be loaded from there. Otherwise, they will be downloaded from Hugging Face.

In [ ]:
print("Loading teacher model...")

teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_NAME,
    cache_dir=MODELS_DIR,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
)
teacher_tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_MODEL_NAME, cache_dir=MODELS_DIR
)

teacher_model.eval()

Loading teacher model...


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
print("Loading student model...")

student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_NAME,
    cache_dir=MODELS_DIR,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
)
student_tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL_NAME, cache_dir=MODELS_DIR
)

print("Student model loaded.")

Loading student model...
Student model loaded.


In [ ]:
# Set pad tokens
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

In [ ]:
print(f"Teacher model: {TEACHER_MODEL_NAME}")
print(f"Student model: {STUDENT_MODEL_NAME}")

Teacher model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Student model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### Loading Private/Gated Models (Optional)

If you need to load private models, uncomment and run:

In [ ]:
# from huggingface_hub import login
#
# # Login to Hugging Face (you'll need a token with access to the model)
# login(token="YOUR_HF_TOKEN_HERE")
#
# # Then load your private model
# TEACHER_MODEL_NAME = "your-org/sleeper-proxy-tinyllama-1.1b"
# teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
# teacher_model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL_NAME, device_map="auto")